# Hydranomic - Gestão Autônoma de Metas e Configurações (WEGnology)

Notebook 100% autônomo e autocontido para execução na nuvem da **WEGnology**.
- Recebe contexto do endpoint via `context.json` gerado pelo template `{{jsonEncode data.body}}`.
- Gerencia criação de metas (`consumo-mensal`, `reducao_mensal`, `gasto-maximo-mensal`, `economia-mensal`) preservando todos os tipos simultaneamente.
- Salva preferências de notificações globais e personalizadas por usuário administrador (`notificacoes_usuarios`).
- Gera e persiste os artefatos `config.json` e `progresso_meta.json` no `OUTPUT_DIR` para sincronização com o **Application Files**.

In [ ]:
# Bloco 1: Adaptador de Ambiente WEGnology (Autocontido sem dependências externas)
import os
import sys
import json
import calendar
from pathlib import Path
from typing import Any, Dict, List, Optional
from datetime import datetime

def obter_diretorio_input() -> Path:
    if 'INPUT_DIR' in os.environ and os.path.isdir(os.environ['INPUT_DIR']):
        return Path(os.environ['INPUT_DIR'])
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / 'workflows').is_dir():
            return p / 'workflows'
        if (p / 'hydranomic' / 'workflows').is_dir():
            return p / 'hydranomic' / 'workflows'
    return Path.cwd()

def obter_diretorio_output() -> Path:
    if 'OUTPUT_DIR' in os.environ:
        caminho = Path(os.environ['OUTPUT_DIR'])
        caminho.mkdir(parents=True, exist_ok=True)
        return caminho
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / 'workflows').is_dir():
            return p / 'workflows'
        if (p / 'hydranomic' / 'workflows').is_dir():
            return p / 'hydranomic' / 'workflows'
    caminho = Path.cwd() / 'outputs'
    caminho.mkdir(parents=True, exist_ok=True)
    return caminho

def carregar_json(nome_arquivo: str, padrao: Any = None) -> Any:
    dir_in = obter_diretorio_input()
    candidatos = [
        dir_in / nome_arquivo,
        Path.cwd() / 'workflows' / nome_arquivo,
        Path.cwd() / 'hydranomic' / 'workflows' / nome_arquivo,
        Path.cwd() / nome_arquivo
    ]
    for c in candidatos:
        if c.exists() and c.stat().st_size > 0:
            with open(c, 'r', encoding='utf-8') as f:
                return json.load(f)
    if padrao is not None:
        return padrao
    raise FileNotFoundError(f'Arquivo {nome_arquivo} nao encontrado em {dir_in}')

def salvar_json(nome_arquivo: str, dados: Any, indent: int = 2) -> Path:
    dir_out = obter_diretorio_output()
    caminho = dir_out / nome_arquivo
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(dados, f, ensure_ascii=False, indent=indent)
    return caminho

def carregar_contexto() -> Dict[str, Any]:
    try:
        return carregar_json('context.json', padrao={})
    except Exception:
        return {}

print('✓ Ambiente WEGnology pronto!')
print('  - INPUT_DIR: ', obter_diretorio_input())
print('  - OUTPUT_DIR:', obter_diretorio_output())


---
## Bloco 2: Parâmetros Injetáveis (Papermill / WEGnology Workflow)
*(Esta célula pode ser sobrescrita automaticamente durante a execução agendada ou via webhook na WEGnology)*

In [ ]:
ESTADO = 'metas'
CONSUMO_ATUAL_M3 = None
CONSUMO_MEDIDO_M3 = None
PERSISTIR_ARQUIVOS = True


---
## Bloco 3: Funções de Gestão de Metas e Configurações (Autocontidas)

In [ ]:
NOME_ARQUIVO_CONFIG = 'config.json'
NOME_ARQUIVO_PROGRESSO = 'progresso_meta.json'

TIPOS_CONSUMO = ('consumo-mensal', 'reducao_mensal')
TIPOS_FINANCEIROS = ('gasto-maximo-mensal', 'economia-mensal')

def obter_config_padrao() -> Dict[str, Any]:
    return {
        'meta_ativa': None,
        'metas_ativas_por_tipo': {},
        'metas_por_mes': {},
        'solicitacao_exclusao': None,
        'notificacoes': {
            'notificacao_prazos': True,
            'alerta_status': True
        },
        'notificacoes_usuarios': {}
    }

def carregar_config(nome_arquivo: str = NOME_ARQUIVO_CONFIG) -> Dict[str, Any]:
    try:
        config = carregar_json(nome_arquivo, padrao=obter_config_padrao())
    except Exception:
        config = obter_config_padrao()
    if not isinstance(config, dict):
        config = obter_config_padrao()

    config.setdefault('meta_ativa', None)
    if not isinstance(config.get('metas_ativas_por_tipo'), dict):
        config['metas_ativas_por_tipo'] = {}
    if not isinstance(config.get('metas_por_mes'), dict):
        config['metas_por_mes'] = {}
    config.setdefault('solicitacao_exclusao', None)

    if 'notificacoes' not in config or not isinstance(config['notificacoes'], dict):
        config['notificacoes'] = {'notificacao_prazos': True, 'alerta_status': True}
    config['notificacoes'].setdefault('notificacao_prazos', True)
    config['notificacoes'].setdefault('alerta_status', True)

    if not isinstance(config.get('notificacoes_usuarios'), dict):
        config['notificacoes_usuarios'] = {}

    # Sanitização: meta_ativa NUNCA pode ser financeira (ex: gasto-maximo-mensal ou economia-mensal)
    meta_ativa = config.get('meta_ativa')
    if isinstance(meta_ativa, dict):
        tipo_ma = str(meta_ativa.get('tipo', '')).strip()
        if tipo_ma in TIPOS_FINANCEIROS:
            if tipo_ma not in config['metas_ativas_por_tipo']:
                config['metas_ativas_por_tipo'][tipo_ma] = dict(meta_ativa)
            c_cons = config['metas_ativas_por_tipo'].get('consumo-mensal') or config['metas_ativas_por_tipo'].get('reducao_mensal')
            config['meta_ativa'] = dict(c_cons) if isinstance(c_cons, dict) else None
        elif tipo_ma in TIPOS_CONSUMO:
            if tipo_ma not in config['metas_ativas_por_tipo']:
                config['metas_ativas_por_tipo'][tipo_ma] = dict(meta_ativa)

    if not config.get('meta_ativa'):
        for tc in TIPOS_CONSUMO:
            if tc in config['metas_ativas_por_tipo'] and isinstance(config['metas_ativas_por_tipo'][tc], dict):
                config['meta_ativa'] = dict(config['metas_ativas_por_tipo'][tc])
                break

    return config

def salvar_config(config: Dict[str, Any], nome_arquivo: str = NOME_ARQUIVO_CONFIG) -> Path:
    return salvar_json(nome_arquivo, config)

def avaliar_progresso_meta(consumo_atual_m3: Optional[float] = None, salvar: bool = True) -> Dict[str, Any]:
    config = carregar_config()

    # Processa se o contexto recebido contiver acao de notificacoes ou metas
    try:
        ctx = carregar_contexto()
        if isinstance(ctx, dict):
            if 'body' in ctx and isinstance(ctx['body'], dict):
                for k, v in ctx['body'].items():
                    if k not in ctx:
                        ctx[k] = v

            # 1. Tratamento de preferências de Notificação
            if ctx.get('acao') == 'salvar_notificacoes' or 'notificacao_prazos' in ctx or 'alerta_status' in ctx or 'notificacoes' in ctx:
                notifs = ctx.get('notificacoes')
                alterou = False
                if isinstance(notifs, dict):
                    if 'notificacao_prazos' in notifs:
                        config['notificacoes']['notificacao_prazos'] = bool(notifs['notificacao_prazos'])
                        alterou = True
                    if 'alerta_status' in notifs:
                        config['notificacoes']['alerta_status'] = bool(notifs['alerta_status'])
                        alterou = True
                if 'notificacao_prazos' in ctx:
                    config['notificacoes']['notificacao_prazos'] = bool(ctx['notificacao_prazos'])
                    alterou = True
                if 'alerta_status' in ctx:
                    config['notificacoes']['alerta_status'] = bool(ctx['alerta_status'])
                    alterou = True

                pref_usr = {
                    'notificacao_prazos': bool(config['notificacoes']['notificacao_prazos']),
                    'alerta_status': bool(config['notificacoes']['alerta_status'])
                }
                chaves_usr = []
                for campo_u in ('usuario', 'nome_usuario', 'email', 'user_id'):
                    val_u = str(ctx.get(campo_u) or '').strip().lower()
                    if val_u:
                        chaves_usr.append(val_u)
                        if '@' in val_u:
                            chaves_usr.append(val_u.split('@')[0].strip())
                if isinstance(ctx.get('chaves_usuario'), list):
                    for cu in ctx['chaves_usuario']:
                        if cu:
                            chaves_usr.append(str(cu).strip().lower())
                for cu in set(chaves_usr):
                    if cu:
                        config['notificacoes_usuarios'][cu] = dict(pref_usr)
                        alterou = True
                if alterou and salvar:
                    salvar_config(config)

            # 2. Definição / Criação de Meta via Contexto
            if (ctx.get('acao') == 'definir_meta' or ctx.get('nome')) and ctx.get('valor') is not None:
                periodo_meta = str(ctx.get('periodo') or ctx.get('data') or datetime.now().strftime('%Y-%m')).strip()
                mes_ref = periodo_meta[:7] if len(periodo_meta) >= 7 else datetime.now().strftime('%Y-%m')
                tipo_meta = str(ctx.get('tipo') or 'consumo-mensal').strip()
                valor_meta = float(ctx.get('valor'))
                eh_consumo = tipo_meta in TIPOS_CONSUMO
                valor_litros = float(ctx.get('valor_litros')) if ctx.get('valor_litros') is not None else (valor_meta * 1000.0 if eh_consumo else None)

                nova_meta_obj = {
                    'tipo': tipo_meta,
                    'unidade': ctx.get('unidade') or ('L' if eh_consumo else 'R$'),
                    'nome': ctx.get('nome', 'Meta de Consumo' if eh_consumo else ('Gasto Máximo' if tipo_meta == 'gasto-maximo-mensal' else 'Meta de Economia')),
                    'valor': valor_meta,
                    'valor_litros': valor_litros,
                    'periodo': periodo_meta,
                    'mes_referencia': mes_ref,
                    'data_criacao': datetime.now().strftime('%Y-%m-%d'),
                    'descricao': ctx.get('descricao', ''),
                    'status': 'ativa'
                }

                # Preservação estrita: grava no tipo correspondente sem sobrescrever os outros
                if not isinstance(config.get('metas_ativas_por_tipo'), dict):
                    config['metas_ativas_por_tipo'] = {}
                config['metas_ativas_por_tipo'][tipo_meta] = nova_meta_obj

                # Apenas metas de consumo (físico em m³) assumem 'meta_ativa'
                if eh_consumo:
                    config['meta_ativa'] = nova_meta_obj

                # Atualiza estrutura no mês preservando os outros tipos
                if not isinstance(config.get('metas_por_mes'), dict):
                    config['metas_por_mes'] = {}
                reg_mes = config['metas_por_mes'].get(mes_ref)
                if not isinstance(reg_mes, dict):
                    reg_mes = {}
                if not isinstance(reg_mes.get('por_tipo'), dict):
                    reg_mes['por_tipo'] = {}
                reg_mes['por_tipo'][tipo_meta] = nova_meta_obj

                # Na raiz do mês, atualiza consumo prioritariamente
                if eh_consumo:
                    reg_mes['nome'] = nova_meta_obj['nome']
                    reg_mes['valor'] = nova_meta_obj['valor']
                    reg_mes['valor_litros'] = valor_litros
                    reg_mes['periodo'] = periodo_meta
                    reg_mes['tipo'] = tipo_meta
                elif 'valor' not in reg_mes:
                    reg_mes['nome'] = nova_meta_obj['nome']
                    reg_mes['valor'] = nova_meta_obj['valor']
                    reg_mes['periodo'] = periodo_meta
                    reg_mes['tipo'] = tipo_meta

                config['metas_por_mes'][mes_ref] = reg_mes
                config['solicitacao_exclusao'] = None
                if salvar:
                    salvar_config(config)
    except Exception as e:
        print(f'Aviso no processamento do contexto: {e}')

    # Leitura do consumo atual acumulado se não informado diretamente
    if consumo_atual_m3 is None:
        try:
            consumo_mes = carregar_json('consumo_mes.json', padrao={})
            consumo_atual_m3 = float(consumo_mes.get('consumo_acumulado_mes_m3', 0.0))
        except Exception:
            try:
                dados_conta = carregar_json('conta.json', padrao=[])
                if isinstance(dados_conta, list) and len(dados_conta) > 0:
                    consumo_atual_m3 = float(dados_conta[-1].get('consumo_m3', 0.0))
                else:
                    consumo_atual_m3 = 0.0
            except Exception:
                consumo_atual_m3 = 0.0

    consumo_val = float(consumo_atual_m3 or 0.0)

    # Identifica meta de consumo ativa para avaliação
    meta = config.get('meta_ativa')
    if not meta or meta.get('tipo') in TIPOS_FINANCEIROS:
        for tc in TIPOS_CONSUMO:
            if tc in config.get('metas_ativas_por_tipo', {}):
                meta = config['metas_ativas_por_tipo'][tc]
                break

    hoje = datetime.now()
    ultimo_dia_mes = calendar.monthrange(hoje.year, hoje.month)[1]
    dias_restantes = max(0, ultimo_dia_mes - hoje.day)

    conf_notif = config.get('notificacoes', {'notificacao_prazos': True, 'alerta_status': True})
    permite_status = conf_notif.get('alerta_status', True)
    permite_prazos = conf_notif.get('notificacao_prazos', True)

    alertas_ativos = []

    if not meta or meta.get('tipo') in TIPOS_FINANCEIROS:
        situacao = 'NORMAL'
        relatorio = {
            'meta_ativa': False,
            'mensagem': 'Nenhuma meta de consumo ativa configurada no momento.',
            'consumo_atual_m3': consumo_val,
            'situacao': situacao,
            'metas_ativas_por_tipo': config.get('metas_ativas_por_tipo', {})
        }
    else:
        limite = float(meta.get('valor', 1.0))
        porcentagem = round((consumo_val / limite) * 100, 2) if limite > 0 else 0.0
        saldo_restante = round(limite - consumo_val, 2)
        if porcentagem >= 100:
            situacao = 'ESTOURADA'
        elif porcentagem >= 80:
            situacao = 'ALERTA_80'
        else:
            situacao = 'NORMAL'

        relatorio = {
            'meta_ativa': True,
            'meta_nome': meta.get('nome', 'Meta de Consumo'),
            'tipo': meta.get('tipo', 'consumo-mensal'),
            'limite_m3': limite,
            'consumo_atual_m3': consumo_val,
            'percentual_atingido': porcentagem,
            'saldo_restante_m3': saldo_restante,
            'situacao': situacao,
            'periodo': meta.get('periodo', ''),
            'mes_referencia': meta.get('mes_referencia', str(meta.get('periodo', ''))[:7]),
            'metas_ativas_por_tipo': config.get('metas_ativas_por_tipo', {})
        }

        if permite_status:
            if situacao == 'ESTOURADA':
                alertas_ativos.append({
                    'id': 'meta_estourada',
                    'tipo': 'limite_estourado',
                    'severidade': 'perigo',
                    'titulo': 'Meta Ultrapassada!',
                    'mensagem': f'O consumo de {consumo_val:.2f} m³ ultrapassou a meta de {limite:.2f} m³ ({porcentagem:.1f}% atingido).',
                    'timestamp': hoje.isoformat()
                })
            elif situacao == 'ALERTA_80':
                alertas_ativos.append({
                    'id': 'meta_80',
                    'tipo': 'alerta_80',
                    'severidade': 'alerta',
                    'titulo': 'Atenção: 80% da Meta Atingida',
                    'mensagem': f'O consumo atingiu {porcentagem:.1f}% da meta estabelecida ({consumo_val:.2f} m³ de {limite:.2f} m³).',
                    'timestamp': hoje.isoformat()
                })

    if permite_prazos and dias_restantes <= 2:
        msg_prazo = 'Faltam menos de 24h para o encerramento do ciclo mensal da sua meta.' if dias_restantes <= 1 else f'Faltam {dias_restantes} dias para o encerramento do ciclo mensal da sua meta.'
        alertas_ativos.append({
            'id': 'prazo_fim_mes',
            'tipo': 'prazo_proximo',
            'severidade': 'info',
            'titulo': 'Prazo de Fechamento Próximo',
            'mensagem': msg_prazo,
            'timestamp': hoje.isoformat()
        })

    relatorio['alertas_ativos'] = alertas_ativos
    relatorio['notificacoes'] = conf_notif
    relatorio['notificacoes_config'] = conf_notif
    relatorio['notificacoes_usuarios'] = config.get('notificacoes_usuarios', {})
    relatorio['dias_restantes_mes'] = dias_restantes

    if salvar:
        salvar_json(NOME_ARQUIVO_PROGRESSO, relatorio)
    return relatorio

print('✓ Funções de gestão de metas carregadas com sucesso!')


---
## Bloco 4: Execução da Avaliação e Persistência no `OUTPUT_DIR`

In [ ]:
contexto_ativo = carregar_contexto()
print(f'Contexto recebido (context.json): {len(contexto_ativo)} propriedades.')
if contexto_ativo:
    acao = contexto_ativo.get('acao') or (contexto_ativo.get('body', {}).get('acao') if isinstance(contexto_ativo.get('body'), dict) else None)
    print(f' - Ação executada: {acao}')

consumo_avaliar = CONSUMO_MEDIDO_M3 if CONSUMO_MEDIDO_M3 is not None else CONSUMO_ATUAL_M3
relatorio_meta = avaliar_progresso_meta(consumo_atual_m3=consumo_avaliar, salvar=PERSISTIR_ARQUIVOS)

# Garante que config.json atualizado é persistido no OUTPUT_DIR para sincronização com Application Files
if PERSISTIR_ARQUIVOS:
    cfg_atual = carregar_config()
    salvar_config(cfg_atual)
    print('✓ config.json gravado no OUTPUT_DIR com sucesso!')
    print('✓ progresso_meta.json gravado no OUTPUT_DIR com sucesso!')


---
## Bloco 5: Auditoria de Saídas e Diagnóstico

In [ ]:
cfg = carregar_config()
print('=' * 60)
print('            STATUS CONSOLIDADO DE METAS E CONFIGURAÇÕES       ')
print('=' * 60)
print('1. Meta Física Ativa (meta_ativa):')
print('   ', cfg.get('meta_ativa'))
print('\n2. Metas Ativas por Tipo (metas_ativas_por_tipo):')
for t, m in cfg.get('metas_ativas_por_tipo', {}).items():
    print(f"   - [{t}]: {m.get('nome')} | Valor: {m.get('valor')} {m.get('unidade')} | Vigência: {m.get('periodo')}")
print('\n3. Preferências de Notificação por Administrador:')
for u, p in cfg.get('notificacoes_usuarios', {}).items():
    print(f'   - {u}: {p}')
print('\n4. Alertas Ativos:')
alertas = relatorio_meta.get('alertas_ativos', []) if isinstance(relatorio_meta, dict) else []
if alertas:
    for a in alertas:
        print(f"   ! [{a.get('severidade', 'info').upper()}] {a.get('titulo')}: {a.get('mensagem')}")
else:
    print('   Nenhum alerta ativo no momento.')

print('\n5. Arquivos persistidos no OUTPUT_DIR:')
dir_out = obter_diretorio_output()
for arq in sorted(dir_out.glob('*.json')):
    print(f'   - {arq.name} ({arq.stat().st_size} bytes)')
print('=' * 60)
